# dotLLM — dual-CUDA cross-device KV-handoff validation (#361)

Validates `StagedKvHandoffTransfer` (#360) across **two T4 GPUs** on Kaggle.

**Before running:** Settings → Accelerator → **GPU T4 ×2**, and Settings → Internet → **On**.

Run the cells top-to-bottom. `test-cpu` proves the seam + .NET 10 toolchain; `test-cuda` is the
dual-device goal (needs the #361 CUDA impl — see `kaggle/README.md`).

In [ ]:
%%bash
# Bootstrap: clone the repo so kaggle/setup.sh is available. Override branch with DOTLLM_BRANCH.
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/361-kaggle-dual-cuda-validation}"
cd /kaggle/working
rm -rf dotLLM
git clone --depth 1 --branch "$DOTLLM_BRANCH" https://github.com/kkokosa/dotLLM.git
echo "cloned $DOTLLM_BRANCH"

In [ ]:
%%bash
# Sanity: OS, dual T4s, nvcc.
bash /kaggle/working/dotLLM/kaggle/setup.sh env

In [ ]:
%%bash
# Install .NET 10 SDK into $HOME (no root).
bash /kaggle/working/dotLLM/kaggle/setup.sh dotnet

In [ ]:
%%bash
# Compile CUDA kernels -> PTX (compute_75; T4 = sm_75).
bash /kaggle/working/dotLLM/kaggle/setup.sh ptx

In [ ]:
%%bash
# Restore + build the solution (Release).
bash /kaggle/working/dotLLM/kaggle/setup.sh build

In [ ]:
%%bash
# CPU parity tests — proves the staged seam + .NET 10 work on Kaggle (passes today).
bash /kaggle/working/dotLLM/kaggle/setup.sh test-cpu

## Dual-device CUDA validation

Runs the cross-device parity test (prefill GPU0 → decode GPU1, token-identical to single-device).
Requires the #361 CUDA `IHostStagedKvCache` impl + `CudaCrossDeviceKvTransferTests` (skips if < 2 GPUs).

In [ ]:
%%bash
bash /kaggle/working/dotLLM/kaggle/setup.sh test-cuda